# Chapter 8 -- Skills, Progressive Disclosure & Code Execution (Practice)

Work through this notebook **after reading** `notes/ch08-skills-and-code-execution.md`. This chapter builds two real Agent Skills on disk (one document-generation, one repo-convention), a genuine `run_python` sandbox tool, and solves one small task two ways -- tool-by-tool versus a single generated program -- measuring real token totals for both, side by side. It then reproduces the notes' Section 11 formulas in code (50 capabilities, three designs) to confirm the ~50x gap by calculation, not just by hand.

Two exercises below have a stub to fill in: **skill activation scoring** (write three Level-1 descriptions and measure activation accuracy across 10 prompts) and **the code-mode token accounting** for Section 11's Design (c). Everything is fully offline and deterministic -- no API key needed for either exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## Part 1 -- Two Real Skills, Written to Disk

Notes Section 2's three-level structure, made concrete: two real `SKILL.md` files, each with a Level-1 description (the trigger), a Level-2 body (workflow steps, output format, definition of done -- Section 4), and a bundled Level-3 resource file that only gets read if the skill's own instructions point at it. One is document-generation (`csv-report`), one is a repo-convention check (`print-narration-check`, enforcing this very repo's own CLAUDE.md print-narration rule).

In [ ]:
SKILLS_ROOT = Path("skills")

CSV_REPORT_SKILL_MD = '''---
name: csv-report
description: Generate a short markdown summary report (totals, averages, one-line takeaway) from a small table of numeric data. Use this when asked to summarize or report on tabular/CSV-style data, not for open-ended data analysis.
---

## Workflow

1. Parse the input rows into a list of numbers.
2. Compute total and average.
3. Fill in `templates/report_template.md` with the computed values.
4. Return the filled template as the final report -- do not add extra commentary outside the template's structure.

## Output Format

A markdown document following exactly the structure in `templates/report_template.md`: a title, a totals line, an average line, and a one-sentence takeaway.

## Definition of Done

The report contains a total, an average, and a takeaway sentence, and no numbers in it were invented -- every figure traces back to the input rows.

## Quality Checks

Common mistake: computing the average over the wrong count (e.g. counting a header row as data). Always confirm the row count excludes headers before dividing.
'''

PRINT_NARRATION_SKILL_MD = '''---
name: print-narration-check
description: Check whether a Python code cell or script follows this repo's mandatory print-narration convention (every non-trivial computation prints a labeled banner and the resulting shapes/values). Use this when reviewing or writing educational code for this repository, not for reviewing production code elsewhere.
---

## Workflow

1. Read `checklist.md` for the exact rules being checked.
2. Scan the code for computations with no adjacent print statement.
3. Flag each one with the line and a suggested print statement.

## Output Format

A short list: one bullet per violation, each naming the line and the missing print.

## Definition of Done

Every non-trivial computation (anything that isn't a pure constant assignment) has an adjacent print statement narrating what it computed, per `checklist.md`.

## Quality Checks

Common mistake: flagging print statements that exist but use `====` banners instead of this repo's `----` convention (see the repo's own feedback memory on banner style) -- that's a style nit, not a missing-narration violation. Don't conflate the two.
'''

REPORT_TEMPLATE = '''# Report

**Total:** {total}
**Average:** {average}

{takeaway}
'''

CHECKLIST = '''# Print-Narration Checklist

- Every tensor/array operation prints its resulting shape.
- Every loop step that changes state prints a labeled line.
- Banners use dash separators (----), not equals signs (====).
'''


def write_skill(name, skill_md, resources):
    skill_dir = SKILLS_ROOT / name
    (skill_dir / "templates").mkdir(parents=True, exist_ok=True)
    (skill_dir / "SKILL.md").write_text(skill_md)
    for rel_path, content in resources.items():
        (skill_dir / rel_path).write_text(content)
    return skill_dir


csv_report_dir = write_skill("csv-report", CSV_REPORT_SKILL_MD, {"templates/report_template.md": REPORT_TEMPLATE})
narration_dir = write_skill("print-narration-check", PRINT_NARRATION_SKILL_MD, {"checklist.md": CHECKLIST})

print(f"Wrote skill: {csv_report_dir}")
print(f"Wrote skill: {narration_dir}")


In [ ]:
def skill_levels(skill_dir):
    """
    Report the byte-size of each of Section 2's three levels for a skill --
    a crude but honest proxy for tokens (roughly 4 bytes/token in English text).
    """
    skill_md = (skill_dir / "SKILL.md").read_text()
    frontmatter_end = skill_md.index("---", 3) + 3
    level1_text = skill_md[:frontmatter_end]  # name + description live in the frontmatter
    level2_text = skill_md[frontmatter_end:]  # everything else: workflow, format, DoD, quality checks
    level3_files = [p for p in skill_dir.rglob("*") if p.is_file() and p.name != "SKILL.md"]
    level3_bytes = sum(p.stat().st_size for p in level3_files)
    return {
        "level1_bytes": len(level1_text),
        "level2_bytes": len(level2_text),
        "level3_bytes": level3_bytes,
        "level3_files": [str(p.relative_to(skill_dir)) for p in level3_files],
    }


print("-" * 60)
print("SKILL LEVEL BREAKDOWN (byte counts as a token proxy, ~4 bytes/token)")
print("-" * 60)
for skill_dir in (csv_report_dir, narration_dir):
    levels = skill_levels(skill_dir)
    print(f"\n{skill_dir.name}:")
    print(f"  Level 1 (always resident):  {levels['level1_bytes']:4d} bytes (~{levels['level1_bytes'] // 4} tokens)")
    print(f"  Level 2 (on activation):    {levels['level2_bytes']:4d} bytes (~{levels['level2_bytes'] // 4} tokens)")
    print(f"  Level 3 (on demand):        {levels['level3_bytes']:4d} bytes across {levels['level3_files']}")


## Part 2 -- Solving One Task Two Ways: Tool-by-Tool vs Code Mode

Same task, same underlying data, two execution shapes (notes Sections 6-7). `estimate_tokens` is a simple, honest proxy (`len(json.dumps(obj)) // 4`) -- not a real tokenizer, but consistent across both ways, which is all a fair side-by-side comparison needs. **Tool-by-tool** calls three separate tools (`get_sales_data`, `compute_stats`, `format_report`), each result round-tripping through the message list. **Code mode** calls one `run_python` tool with a single generated script that does all three steps internally and returns only the final report string.

In [ ]:
import json


def estimate_tokens(obj):
    """Crude, consistent token proxy: ~4 bytes per token, applied to the JSON-serialized form."""
    return len(json.dumps(obj)) // 4


SALES_DATA = [120, 95, 140, 110, 130]


def get_sales_data():
    return SALES_DATA


def compute_stats(data):
    return {"total": sum(data), "average": sum(data) / len(data)}


def format_report(total, average):
    return f"Total sales: {total}. Average: {average:.1f}."


In [ ]:
def run_tool_by_tool():
    """Three separate tool round-trips -- each schema declared, each result appended to messages."""
    tool_schemas = [
        {"name": "get_sales_data", "description": "Return the raw sales figures.", "input_schema": {"type": "object", "properties": {}}},
        {"name": "compute_stats", "description": "Compute total and average of a list of numbers.",
         "input_schema": {"type": "object", "properties": {"data": {"type": "array"}}}},
        {"name": "format_report", "description": "Format a one-line sales report.",
         "input_schema": {"type": "object", "properties": {"total": {"type": "number"}, "average": {"type": "number"}}}},
    ]
    messages = [{"role": "user", "content": "Give me a one-line sales report."}]
    total_tokens = estimate_tokens(tool_schemas) + estimate_tokens(messages)

    # Step 1: get_sales_data
    data = get_sales_data()
    messages.append({"role": "assistant", "content": [{"type": "tool_use", "id": "c1", "name": "get_sales_data", "input": {}}]})
    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": "c1", "content": data}]})
    total_tokens += estimate_tokens(tool_schemas) + estimate_tokens(messages)

    # Step 2: compute_stats
    stats = compute_stats(data)
    messages.append({"role": "assistant", "content": [{"type": "tool_use", "id": "c2", "name": "compute_stats", "input": {"data": data}}]})
    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": "c2", "content": stats}]})
    total_tokens += estimate_tokens(tool_schemas) + estimate_tokens(messages)

    # Step 3: format_report
    report = format_report(stats["total"], stats["average"])
    messages.append({"role": "assistant", "content": [{"type": "tool_use", "id": "c3", "name": "format_report", "input": stats}]})
    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": "c3", "content": report}]})
    total_tokens += estimate_tokens(tool_schemas) + estimate_tokens(messages)

    return report, total_tokens, 3  # 3 round trips


def run_code_mode():
    """One tool (run_python), one generated script, one round trip."""
    run_python_schema = [{"name": "run_python", "description": "Execute a Python script and return its printed output.",
                           "input_schema": {"type": "object", "properties": {"code": {"type": "string"}}}}]
    messages = [{"role": "user", "content": "Give me a one-line sales report."}]
    total_tokens = estimate_tokens(run_python_schema) + estimate_tokens(messages)

    generated_code = (
        "data = get_sales_data()\n"
        "stats = compute_stats(data)\n"
        "print(format_report(stats['total'], stats['average']))"
    )
    # Real execution, in-process (a genuine sandbox would isolate this -- Section 8) --
    # the point being demonstrated here is the token accounting, not sandbox isolation.
    namespace = {"get_sales_data": get_sales_data, "compute_stats": compute_stats, "format_report": format_report}
    import io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        exec(generated_code, namespace)
    report = buf.getvalue().strip()

    messages.append({"role": "assistant", "content": [{"type": "tool_use", "id": "c1", "name": "run_python", "input": {"code": generated_code}}]})
    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": "c1", "content": report}]})
    total_tokens += estimate_tokens(run_python_schema) + estimate_tokens(messages)

    return report, total_tokens, 1  # 1 round trip


report_a, tokens_a, trips_a = run_tool_by_tool()
report_b, tokens_b, trips_b = run_code_mode()

print("-" * 60)
print("SAME TASK, TWO WAYS")
print("-" * 60)
print(f"Tool-by-tool : report={report_a!r}  round_trips={trips_a}  tokens~={tokens_a}")
print(f"Code mode    : report={report_b!r}  round_trips={trips_b}  tokens~={tokens_b}")
print(f"\nToken ratio (tool-by-tool / code mode): {tokens_a / tokens_b:.2f}x")

assert report_a == report_b, "both ways must produce the identical final report"
assert tokens_b < tokens_a, "code mode should cost fewer estimated tokens on this task"
print("\nConfirmed: identical final report, fewer round trips and fewer estimated")
print("tokens for code mode -- the mechanism from notes Section 6, not just the formula.")


## Part 3 -- Reproducing Notes Section 11's Formulas in Code

The three-design comparison from the notes (50 capabilities, 20 steps), computed here instead of by hand, to confirm the arithmetic is exactly reproducible -- not an artifact of hand-arithmetic rounding.

In [ ]:
def design_a_all_resident(n_capabilities=50, n_steps=20, tokens_per_schema=250):
    per_step = n_capabilities * tokens_per_schema
    return per_step, per_step * n_steps


def design_b_skills(n_capabilities=50, n_steps=20, level1_tokens=100,
                     activations=((3, 3000), (10, 2500))):
    """activations: list of (activation_step, level2_body_tokens), 1-indexed steps, sorted ascending."""
    standing = n_capabilities * level1_tokens
    total = 0
    resident_bodies = 0
    activations = sorted(activations)
    next_activation_idx = 0
    per_step_trace = []
    for step in range(1, n_steps + 1):
        while next_activation_idx < len(activations) and activations[next_activation_idx][0] == step:
            resident_bodies += activations[next_activation_idx][1]
            next_activation_idx += 1
        this_step_tokens = standing + resident_bodies
        total += this_step_tokens
        per_step_trace.append(this_step_tokens)
    return per_step_trace, total


a_per_step, a_total = design_a_all_resident()
b_trace, b_total = design_b_skills()
print(f"Design (a) all resident: {a_per_step} tokens/step (flat) -> total {a_total}")
assert a_total == 250_000, f"expected 250,000 to match the notes, got {a_total}"

print(f"Design (b) skills: per-step trace = {b_trace}")
assert b_total == 181_500, f"expected 181,500 to match the notes, got {b_total}"
print(f"Design (b) total: {b_total}")

print("\nBoth match notes Section 11 exactly.")


## Exercise 1 -- Skill Activation Scoring

Implement `activation_score(description, prompt)`: a simple word-overlap heuristic (notes Section 4's whole point -- the description is the *only* information available at activation time, so a real system's matcher works from exactly this little). A prompt is short and task-specific, so score in the direction that actually carries signal: return the fraction of the **prompt's** meaningful words (lowercased, stripped of trailing punctuation, length > 3 to skip stopwords like "the"/"and") that also appear as a substring somewhere in the **description**. Then use it, together with the given `THRESHOLD`, to classify which of three skills (two well-written, one deliberately vague) activates for each of 10 test prompts.

In [ ]:
THRESHOLD = 0.3

SKILL_DESCRIPTIONS = {
    "csv-report": "Generate a short markdown summary report totals averages one-line takeaway from a small table of numeric data. Use this when asked to summarize or report on tabular CSV-style data.",
    "print-narration-check": "Check whether a Python code cell or script follows this repo's mandatory print-narration convention labeled banner printed shapes values. Use this when reviewing or writing educational code for this repository.",
    "helper": "Helps with stuff and things when you need assistance.",  # deliberately vague -- notes Section 4's warning
}

# The first 6 have one clearly correct skill; the last 4 are deliberately
# ambiguous/off-topic -- exactly the prompts a vague description is most
# likely to accidentally fire on.
TEST_PROMPTS = [
    ("Can you summarize this table of numbers into a report?",                 "csv-report"),
    ("I have a CSV of quarterly sales, give me totals and an average.",        "csv-report"),
    ("Does this code cell print the shapes it computes?",                      "print-narration-check"),
    ("Review this notebook for missing print statements per repo convention.", "print-narration-check"),
    ("Summarize the average and total for this table of CSV data.",           "csv-report"),
    ("Check this script for missing print narration banners.",                "print-narration-check"),
    ("What's the capital of France?",                                          None),
    ("Write me a poem about the ocean.",                                       None),
    ("Can you help me with stuff?",                                            None),
    ("I need assistance with things related to my report.",                   None),
]


def activation_score(description, prompt):
    """
    Fraction of the PROMPT's meaningful words (len > 3, trailing punctuation
    stripped) that also appear as a substring somewhere in the description.
    Returns a float in [0, 1].
    """
    # TODO: split `prompt` on whitespace, strip trailing ".,?!" from each
    # word, lowercase it, and keep only words with len > 3 -- call this list
    # `meaningful_words`. If it's empty, return 0.0. Otherwise, lowercase
    # `description` once, count how many of `meaningful_words` appear as a
    # substring of it, and return that count divided by len(meaningful_words).
    meaningful_words = []
    # your code here

    return 0.0


In [ ]:
def classify(prompt):
    """Return the skill name with the highest activation_score above THRESHOLD, or None."""
    scores = {name: activation_score(desc, prompt) for name, desc in SKILL_DESCRIPTIONS.items()}
    best_name, best_score = max(scores.items(), key=lambda kv: kv[1])
    return (best_name if best_score >= THRESHOLD else None), scores


correct = 0
specific_correct = 0
helper_false_positives = 0
print("-" * 60)
print("ACTIVATION ACCURACY ACROSS 10 PROMPTS")
print("-" * 60)
for prompt, expected in TEST_PROMPTS:
    predicted, scores = classify(prompt)
    is_correct = predicted == expected
    correct += is_correct
    if expected is not None:
        specific_correct += is_correct
    if predicted == "helper":
        helper_false_positives += 1
    print(f"  {'OK ' if is_correct else 'ERR'} expected={str(expected):24s} predicted={str(predicted):24s} prompt={prompt!r}")

n_specific = sum(1 for _, expected in TEST_PROMPTS if expected is not None)
specific_accuracy = specific_correct / n_specific
overall_accuracy = correct / len(TEST_PROMPTS)
print(f"\nAccuracy on the 6 prompts with a real target skill: {specific_correct}/{n_specific} = {specific_accuracy:.0%}")
print(f"Overall accuracy (including the 4 ambiguous/off-topic prompts): {correct}/{len(TEST_PROMPTS)} = {overall_accuracy:.0%}")
print(f"'helper' (the deliberately vague skill) fired {helper_false_positives} time(s) on prompts")
print("where NO skill should have activated -- this is notes Section 4's over-firing gotcha,")
print("made measurable instead of asserted away: a vague description costs you exactly here.")

assert specific_accuracy == 1.0, f"expected 100% accuracy on prompts with a real, on-topic target skill, got {specific_accuracy:.0%}"
assert helper_false_positives >= 1, "this exercise's ambiguous prompts are specifically chosen to demonstrate the vague skill CAN over-fire -- if this is 0, the demonstration didn't land"
print("\nExercise 1 PASSED -- concrete, on-topic descriptions win reliably on every prompt")
print("aimed at them, while the vague description demonstrably over-fires on ambiguous")
print("input, exactly as notes Section 4 warns.")


## Exercise 2 -- Design (c): Code Mode's Token Accounting

Implement `design_c_code_mode(n_steps=20, tokens_per_schema=250)`: notes Section 11's third design. Exactly **one** schema (the `run_code` tool) is resident every step -- no other capability is ever declared in context, regardless of how many capabilities the client library actually exposes. Return `(per_step, total)`.

In [ ]:
def design_c_code_mode(n_steps=20, tokens_per_schema=250):
    """
    One schema, flat across every step -- the whole point of code mode is that
    the other 49 capabilities never appear in context at all.
    """
    # TODO: per_step is just `tokens_per_schema` (one resident tool schema,
    # regardless of n_steps or how many capabilities the client library
    # actually exposes). total is per_step * n_steps. Return (per_step, total).
    per_step = 0
    total = 0
    return per_step, total


In [ ]:
c_per_step, c_total = design_c_code_mode()
print(f"Design (c) code mode: {c_per_step} tokens/step (flat) -> total {c_total}")
assert c_total == 5_000, f"expected 5,000 to match the notes, got {c_total}"

ratio = a_total / c_total
print(f"\nDesign (a) / Design (c) ratio: {ratio:.0f}x")
assert ratio == 50, f"expected exactly a 50x ratio, got {ratio}x"

print("\nExercise 2 PASSED -- confirms notes Section 11's headline number: declaring")
print("all 50 capabilities individually costs exactly 50x a single code-mode tool,")
print("which falls directly out of the capability count, not out of any rounding.")


## Optional -- Ask a Real Claude Model Which Skill It Would Activate

The exact same three skill descriptions from Exercise 1, but the activation decision now made by a real model instead of the word-overlap heuristic -- a genuinely live comparison of a simple proxy against the real thing the proxy is standing in for.

In [ ]:
RUN_REAL_ACTIVATION_DEMO = False


def run_real_activation_demo():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        print("Skipping real activation demo: AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env.")
        return

    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    descriptions_block = "\n".join(f"- {name}: {desc}" for name, desc in SKILL_DESCRIPTIONS.items())
    for prompt, expected in TEST_PROMPTS[:4]:
        ask = (
            f"Given these available skills:\n{descriptions_block}\n\n"
            f"Task: \"{prompt}\"\n\n"
            "Which single skill name would you activate, if any? Reply with just the "
            "skill name, or 'none'."
        )
        try:
            response = real_client.messages.create(
                model=MODEL_NAME, max_tokens=20,
                messages=[{"role": "user", "content": ask}],
            )
            reply = next((b.text for b in response.content if b.type == "text"), "").strip()
            print(f"  prompt={prompt!r}  expected={expected}  model_picked={reply!r}")
        except Exception as exc:
            print(f"Real activation demo failed: {type(exc).__name__}: {exc}")
            return


if RUN_REAL_ACTIVATION_DEMO:
    run_real_activation_demo()
else:
    print("RUN_REAL_ACTIVATION_DEMO is False -- running in offline/heuristic mode only.")
    print("Flip it to True to compare against a real Claude model's own activation choices via Bedrock.")


## Key Takeaways

You wrote two real skills to disk with all three of Section 2's levels represented, then solved one task two structurally different ways -- three separate tool round-trips versus one generated program -- and measured, not just asserted, that code mode produces the identical result with fewer round trips and fewer estimated tokens. Reproducing notes Section 11's formulas in code confirmed the headline 50x gap follows directly from capability count, with skills landing at a real but more modest ~1.4x improvement because a conservative "nothing gets evicted" assumption lets activated bodies accumulate. The activation-scoring exercise made Section 4's central claim measurable: a concrete, on-topic description wins reliably, and a vague one never wins anything, using nothing more than word overlap.

**Connection forward:** Chapter 9 turns to a problem this chapter's mechanisms both quietly assumed away -- everything here (skills, code mode, tool search) manages what's visible *within one run*. The moment the run ends, none of it says what, if anything, survives into the next one. That's memory, and it's a genuinely different problem.